# Dataset rescue statuses

Checks the rescue status of the [example data sources](https://docs.google.com/document/d/1rYsP1I1-TAd3Tu-G6EJ-5o7IbQIHf_dKT3DjH8IpQuo/edit?tab=t.0#heading=h.87di2doy0z55).


## Set up the path


In [1]:
import sys
from pathlib import Path

repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "ptps_wildfire_demo").is_dir()
)
sys.path.insert(0, str(repo_root))

In [2]:
import httpx

from ptps_wildfire_demo.proxy.resolver import Resolver

client = httpx.AsyncClient()
resolver = Resolver(client)

## Load datasets


In [3]:
import pandas as pd

datasets = pd.read_csv("fire_datasets.csv")
datasets

,access_type,name,description,webpage,example_data_url
0,No auth,CarbonPlan Open Climate Risk,Building-level wildfire risk.,https://source.coop/carbonplan/carbonplan-ocr,https://s3.us-west-2.amazonaws.com/us-west-2.o...
1,No auth,cboettig fire,Historical burned-area perimeters and hazard p...,https://source.coop/cboettig/fire,https://data.source.coop/cboettig/fire/usgs-mt...
2,No auth,bkr observations,"Weather station, radiosonde, and aviation obse...",https://source.coop/bkr/obs,NaN
3,No auth,giswqs National Wetlands Inventory,National Wetlands Inventory.,https://source.coop/giswqs/nwi,NaN
4,No auth,USFS probabilistic wildfire risk burn probability,Burn probability and flame-length grids.,https://data-usfs.hub.arcgis.com/datasets/usfs...,https://imagery.geoplatform.gov/iipp/rest/serv...
5,No auth,Wildfire Risk to Communities,Wildfire risk data downloads.,https://wildfirerisk.org/download/,https://wildfirerisk.org/wp-content/uploads/20...
6,No auth,NOAA weather alerts,"Forecasts and active alerts, including Red Fla...",https://www.weather.gov/documentation/services...,https://api.weather.gov/alerts/active?event=Re...
7,No auth,NOAA HMS fire and smoke product,Daily smoke plume and fire detection polygons.,https://www.ospo.noaa.gov/products/land/hms.html,https://satepsanone.nesdis.noaa.gov/pub/FIRE/w...
8,No auth,NOAA Storm Events Database,Historical disaster impacts by county.,https://www.ncdc.noaa.gov/stormevents/,https://www.ncei.noaa.gov/pub/data/swdi/storme...
9,No auth,NASA FIRMS KML fire footprints,Regional active-fire footprint polygons.,https://firms.modaps.eosdis.nasa.gov/,https://firms.modaps.eosdis.nasa.gov/data/acti...


## Check status

Only include the official government data sources that don't require auth.


In [4]:
no_auth = datasets["access_type"] != "Free key"
not_src_coop = ~datasets["webpage"].str.startswith("https://source.coop/")
not_dryad = ~datasets["webpage"].str.contains("/dryad.")
has_example_data_url = datasets["example_data_url"].notna()

datasets_to_check = datasets[no_auth & not_src_coop & not_dryad & has_example_data_url]
datasets_to_check

,access_type,name,description,webpage,example_data_url
4,No auth,USFS probabilistic wildfire risk burn probability,Burn probability and flame-length grids.,https://data-usfs.hub.arcgis.com/datasets/usfs...,https://imagery.geoplatform.gov/iipp/rest/serv...
5,No auth,Wildfire Risk to Communities,Wildfire risk data downloads.,https://wildfirerisk.org/download/,https://wildfirerisk.org/wp-content/uploads/20...
6,No auth,NOAA weather alerts,"Forecasts and active alerts, including Red Fla...",https://www.weather.gov/documentation/services...,https://api.weather.gov/alerts/active?event=Re...
7,No auth,NOAA HMS fire and smoke product,Daily smoke plume and fire detection polygons.,https://www.ospo.noaa.gov/products/land/hms.html,https://satepsanone.nesdis.noaa.gov/pub/FIRE/w...
8,No auth,NOAA Storm Events Database,Historical disaster impacts by county.,https://www.ncdc.noaa.gov/stormevents/,https://www.ncei.noaa.gov/pub/data/swdi/storme...
9,No auth,NASA FIRMS KML fire footprints,Regional active-fire footprint polygons.,https://firms.modaps.eosdis.nasa.gov/,https://firms.modaps.eosdis.nasa.gov/data/acti...
10,No auth,WFIGS / National Interagency Fire Center,Current and historical fire perimeters from US...,https://data-nifc.opendata.arcgis.com/,https://gis.blm.gov/arcgis/rest/services/fire/...
11,No auth,MODIS NDVI,Vegetation-health data useful as a live fuel-c...,https://modis.gsfc.nasa.gov/data/dataprod/mod1...,https://cmr.earthdata.nasa.gov/search/granules...
13,No auth,USDA LANDFIRE,"Wildland fire data, including seasonal fuels.",https://www.landfire.gov/,https://www.landfire.gov/data-downloads/Season...
17,Bulk or analytic,CDC PLACES,Census-tract and county chronic-disease preval...,https://www.cdc.gov/places/,https://data.cdc.gov/resource/i46a-9kgh.geojson


In [5]:
import asyncio

from helpers import get_statuses

# essentially an async Series.apply()
statuses = await get_statuses(client, datasets_to_check["example_data_url"])
rescues = await asyncio.gather(
    *(resolver.get_rescue(url) for url in datasets_to_check["example_data_url"])
)

# display(statuses)
# rescues

In [6]:
rescues_df = pd.DataFrame(rescues).rename(columns={"original_url": "example_data_url"})
rescues_df.insert(1, "example_data_url_status", statuses)
results = pd.merge(datasets_to_check, rescues_df, on="example_data_url")

results.drop(columns=["access_type", "webpage", "description", "resolved_url"])

,name,example_data_url,example_data_url_status,wayback_newest_url,drp_url
0,USFS probabilistic wildfire risk burn probability,https://imagery.geoplatform.gov/iipp/rest/serv...,🟢 200,NaN,None
1,Wildfire Risk to Communities,https://wildfirerisk.org/wp-content/uploads/20...,🟢 200,NaN,None
2,NOAA weather alerts,https://api.weather.gov/alerts/active?event=Re...,🟢 200,NaN,None
3,NOAA HMS fire and smoke product,https://satepsanone.nesdis.noaa.gov/pub/FIRE/w...,🟢 200,NaN,None
4,NOAA Storm Events Database,https://www.ncei.noaa.gov/pub/data/swdi/storme...,🟢 200,NaN,None
5,NASA FIRMS KML fire footprints,https://firms.modaps.eosdis.nasa.gov/data/acti...,🟢 200,http://web.archive.org/web/20260110052750/http...,None
6,WFIGS / National Interagency Fire Center,https://gis.blm.gov/arcgis/rest/services/fire/...,🟢 200,http://web.archive.org/web/20260209203352/http...,None
7,MODIS NDVI,https://cmr.earthdata.nasa.gov/search/granules...,🟢 200,NaN,None
8,USDA LANDFIRE,https://www.landfire.gov/data-downloads/Season...,🟢 200,NaN,None
9,CDC PLACES,https://data.cdc.gov/resource/i46a-9kgh.geojson,🟢 200,http://web.archive.org/web/20260315024944/http...,None


### Webpage URLs


In [7]:
page_statuses = await get_statuses(client, datasets["webpage"])
page_rescues = await asyncio.gather(
    *(resolver.get_rescue(url) for url in datasets["webpage"])
)

In [8]:
page_rescues_df = pd.DataFrame(page_rescues)
page_rescues_df.insert(2, "resolved_url_status", page_statuses)

page_rescues_df.drop(columns="resolved_url")

,original_url,resolved_url_status,wayback_newest_url,drp_url
0,https://source.coop/carbonplan/carbonplan-ocr,🟢 200,http://web.archive.org/web/20260412224739/http...,NaN
1,https://source.coop/cboettig/fire,🟢 200,http://web.archive.org/web/20260517053656/http...,NaN
2,https://source.coop/bkr/obs,🟢 200,http://web.archive.org/web/20260815165154/http...,NaN
3,https://source.coop/giswqs/nwi,🟢 200,http://web.archive.org/web/20260803053438/http...,NaN
4,https://data-usfs.hub.arcgis.com/datasets/usfs...,🟢 200,NaN,NaN
5,https://wildfirerisk.org/download/,🟢 200,http://web.archive.org/web/20260805131156/http...,NaN
6,https://www.weather.gov/documentation/services...,🟢 200,http://web.archive.org/web/20260826090451/http...,NaN
7,https://www.ospo.noaa.gov/products/land/hms.html,🟢 200,http://web.archive.org/web/20260729164509/http...,NaN
8,https://www.ncdc.noaa.gov/stormevents/,🟢 200,http://web.archive.org/web/20250810191720/http...,NaN
9,https://firms.modaps.eosdis.nasa.gov/,🟢 200,http://web.archive.org/web/20260822172456/http...,NaN
